In [ ]:
!pip install -q 'peft<0.14.0' transformers>=4.45.0

import importlib
# Force reimport of peft modules to pick up new version
import sys
for mod_name in list(sys.modules.keys()):
    if 'peft' in mod_name:
        del sys.modules[mod_name]
print('Downgraded peft and cleared cache')


# LilTii Banglish Emotion LLM Training

Training Qwen2.5-0.5B with QLoRA on 2000+ Banglish examples.

## Dataset
- 2000 curated CupCake examples from `training/dataset/liltii_fewshot.jsonl`
- 1600 multi-turn conversations from `training/dataset/multiturn_train.jsonl`
- Combined: 3600 examples


In [ ]:
%%capture
!pip install -q transformers peft bitsandbytes datasets accelerate sentencepiece pyyaml huggingface_hub

## Imports & Setup


In [ ]:
import os
import json
import random
from pathlib import Path

import torch
import numpy as np
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Output directory
OUTPUT_DIR = "/kaggle/working/output"
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "liltii-v0.2-lora")
MERGED_DIR = os.path.join(OUTPUT_DIR, "liltii-v0.2-merged")
os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(MERGED_DIR, exist_ok=True)

# GPU check
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Load Dataset


In [ ]:
from datasets import load_dataset

# Load from HuggingFace Hub
print("Loading CupCake dataset from HuggingFace...")
hf_dataset = load_dataset("bayazidhs04/cupcake-liltii")
train_data = hf_dataset['train']
val_data = hf_dataset['validation']

# Convert to list format
all_data = list(train_data) + list(val_data)
random.shuffle(all_data)
print(f"Total examples: {len(all_data)}")

# Split 90/10
split_idx = int(0.9 * len(all_data))
train_data = all_data[:split_idx]
val_data = all_data[split_idx:]

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

# Show sample
print("\nSample:")
sample = train_dataset[0]
for msg in sample['messages']:
    print(f"  {msg['role']}: {msg['content'][:80]}...")


## Train Model


In [ ]:
# Monkey-patch to bypass torchao version check in peft
import importlib.util
spec = importlib.util.find_spec('torchao')
if spec is not None:
    import torchao
    torchao.__version__ = '99.99.99'
    import peft.import_utils as _iu
    _iu.torchao_version = '99.99.99'
    print('Patched torchao version to bypass peft check')
else:
    print('torchao not installed, no patch needed')


In [ ]:
# Model config
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_SEQ_LENGTH = 2048

# Quantization config (4-bit QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

# Load tokenizer
print(f"Loading tokenizer: {BASE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model
print(f"Loading model: {BASE_MODEL}")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# LoRA config
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Tokenize function
def tokenize_fn(example):
    messages = example['messages']
    if tokenizer.chat_template:
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    else:
        parts = [f"{m['role']}\n{m['content']}" for m in messages]
        prompt = "\n".join(parts)
    tokenized = tokenizer(prompt, truncation=True, max_length=MAX_SEQ_LENGTH, padding=False)
    tokenized['labels'] = tokenized['input_ids'][:]
    return tokenized

tokenized_train = train_dataset.map(tokenize_fn, remove_columns=train_dataset.column_names)
tokenized_val = val_dataset.map(tokenize_fn, remove_columns=val_dataset.column_names)
print(f"Tokenized train: {len(tokenized_train)}, val: {len(tokenized_val)}")

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir=ADAPTER_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    seed=SEED,
)

# Data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, padding=True, pad_to_multiple_of=8
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    processing_class=tokenizer,
)

print("\n=== Starting Training ===")
trainer.train()

# Save adapter
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"\nAdapter saved to: {ADAPTER_DIR}")

# Copy to output
import shutil
for fname in os.listdir(ADAPTER_DIR):
    src = os.path.join(ADAPTER_DIR, fname)
    dst = os.path.join(OUTPUT_DIR, fname)
    if os.path.isfile(src):
        shutil.copy2(src, dst)
print("Files copied to /kaggle/working/output/")

## Merge Adapter (Optional)


In [ ]:
# Merge LoRA into base model for easier deployment
from peft import PeftModel

print("Merging LoRA adapter...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
lora_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
merged_model = lora_model.merge_and_unload()
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

print(f"\nMerged model saved to: {MERGED_DIR}")
print("\nOutput files:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / 1e6
        print(f"  {f}: {size:.1f} MB")